# 03 · Delivery performance

**Business questions**

- **Q2.** Which sellers miss their dispatch deadline most often, and is it structural or circumstantial?
- **Q3.** How much of the delay felt by the customer is the seller's, and how much is the logistics leg's?
- **Q4.** How do delay and distance relate to review score and freight cost?

**Definitions.** Unit of analysis is the order of a single-seller order (multi-seller orders excluded, 1.3% of orders), so large orders do not weigh more. *Dispatch on time* = handed to the carrier by the seller's shipping limit (the seller's SLA). *Customer on time* = delivered by the estimated date, compared by calendar day. Window Jan 2017 to Aug 2018. All relationships are associations, not causal effects.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr

from src.data import q
from src.style import C, MONO, finish, save

pd.set_option("display.float_format", "{:,.3f}".format)
BASE = "FROM fct_seller_orders WHERE in_window AND is_single_seller"

## 1 · Monthly SLA and the Black Friday peak (Q2)

In [2]:
monthly = q(f"""
    SELECT DATE_TRUNC('month', purchased_at) AS month,
           COUNT(*) AS orders,
           AVG(dispatch_on_time) AS dispatch_on_time,
           AVG(customer_on_time) AS customer_on_time,
           AVG(CASE WHEN dispatch_on_time IS NULL THEN 1.0 ELSE 0 END) AS pct_without_dispatch
    {BASE} AND is_valid_status
    GROUP BY 1 ORDER BY 1""")
monthly["month"] = pd.to_datetime(monthly["month"])

BF = pd.Timestamp("2017-11-01")
others = monthly[monthly["month"] != BF]
bf = monthly.loc[monthly["month"] == BF].iloc[0]
base_dispatch = others["dispatch_on_time"].median()
base_customer = others["customer_on_time"].median()
bf_drop = base_dispatch - bf["dispatch_on_time"]

print(f"Dispatch on time, median month:  {base_dispatch:.1%}")
print(f"Dispatch on time, Nov 2017:      {bf['dispatch_on_time']:.1%}  ({-bf_drop * 100:+.1f} pp vs median)")
print(f"Customer on time, median month:  {base_customer:.1%}")
print(f"Customer on time, Nov 2017:      {bf['customer_on_time']:.1%}")
print(f"Orders in Nov 2017 vs median:    {bf['orders'] / others['orders'].median():.1f}x")
print("\nCheck for censoring at the end of the window (orders not yet dispatched):")
print(monthly.tail(3)[["month", "orders", "pct_without_dispatch"]].to_string(index=False))

Dispatch on time, median month:  92.2%
Dispatch on time, Nov 2017:      87.0%  (-5.2 pp vs median)
Customer on time, median month:  95.6%
Customer on time, Nov 2017:      87.5%
Orders in Nov 2017 vs median:    1.6x

Check for censoring at the end of the window (orders not yet dispatched):
     month  orders  pct_without_dispatch
2018-06-01    6038                 0.000
2018-07-01    6136                 0.002
2018-08-01    6309                 0.004


In [3]:
fig = go.Figure()
fig.add_scatter(x=monthly["month"], y=monthly["customer_on_time"], mode="lines",
                name="Delivered to customer by estimated date",
                line=dict(color=C["sage_500"], width=2))
fig.add_scatter(x=monthly["month"], y=monthly["dispatch_on_time"], mode="lines",
                name="Dispatched by seller deadline",
                line=dict(color=C["forest_700"], width=2.5))
fig.add_scatter(x=[bf["month"]], y=[bf["dispatch_on_time"]], mode="markers", showlegend=False,
                hoverinfo="skip", marker=dict(symbol="square", size=11, color=C["unfavorable"]))
fig.add_annotation(x=bf["month"], y=bf["dispatch_on_time"], ax=40, ay=50, arrowcolor=C["axis"],
                   arrowwidth=1, xanchor="left", font=dict(size=12, color=C["ink"]),
                   text=f"<b>Nov 2017 · Black Friday</b><br>dispatch on time {bf['dispatch_on_time']:.0%}"
                        f" ({-bf_drop * 100:+.0f} pp vs median month)" if bf_drop > 0 else
                        f"<b>Nov 2017 · Black Friday</b><br>dispatch on time {bf['dispatch_on_time']:.0%}")
lo = min(monthly["dispatch_on_time"].min(), monthly["customer_on_time"].min())
fig.update_yaxes(title="Share of orders on time", tickformat=".0%", range=[max(0, lo - 0.1), 1.0])
fig.update_xaxes(title=None, dtick="M2", tickformat="%b %Y")
title = (f"Black Friday cut on-time dispatch by {bf_drop * 100:.0f} points"
         if bf_drop >= 0.02 else "On-time dispatch held through the Black Friday peak")
finish(fig, title, "Monthly share of single-seller orders on time · seller SLA vs customer promise",)
fig.show()
save(fig, "03_monthly_sla")

WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/03_monthly_sla.png')

## 2 · Structural or circumstantial? (Q2)

Each seller's dispatch on-time rate in 2017 is compared with the same seller in 2018 (the split balances order volume: about 45k and 54k orders). Only sellers with at least 15 dispatched orders in each half are compared. If performance were pure noise, a seller in the bottom quartile in 2017 would stay there in 2018 about 25% of the time.

In [4]:
h = q(f"""
    SELECT seller_id,
           AVG(dispatch_on_time)   FILTER (WHERE purchased_at <  DATE '2018-01-01') AS h1,
           COUNT(dispatch_on_time) FILTER (WHERE purchased_at <  DATE '2018-01-01') AS n1,
           AVG(dispatch_on_time)   FILTER (WHERE purchased_at >= DATE '2018-01-01') AS h2,
           COUNT(dispatch_on_time) FILTER (WHERE purchased_at >= DATE '2018-01-01') AS n2
    {BASE}
    GROUP BY seller_id""")
h = h[(h["n1"] >= 15) & (h["n2"] >= 15)].copy()

rho, pval = spearmanr(h["h1"], h["h2"])
q1_cut, q2_cut = h["h1"].quantile(0.25), h["h2"].quantile(0.25)
h["low_h1"], h["low_h2"] = h["h1"] <= q1_cut, h["h2"] <= q2_cut
h["persistent_laggard"] = h["low_h1"] & h["low_h2"]
stay_rate = h["persistent_laggard"].sum() / h["low_h1"].sum()

print(f"Sellers compared:                         {len(h):,}")
print(f"Spearman correlation 2017 vs 2018:        {rho:.2f} (p = {pval:.1g})")
print(f"Bottom-quartile cut, 2017 / 2018:         {q1_cut:.1%} / {q2_cut:.1%}")
print(f"Bottom quartile in 2017 still there 2018: {stay_rate:.0%} (25% expected by chance)")
print(f"Persistent laggards:                      {int(h['persistent_laggard'].sum())}")

Sellers compared:                         338
Spearman correlation 2017 vs 2018:        0.59 (p = 1e-32)
Bottom-quartile cut, 2017 / 2018:         88.7% / 88.9%
Bottom quartile in 2017 still there 2018: 62% (25% expected by chance)
Persistent laggards:                      53


In [5]:
fig = go.Figure()
for name, mask, color, size in [
    ("Other sellers", ~h["persistent_laggard"], C["context"], 7),
    ("Bottom quartile in both years", h["persistent_laggard"], C["unfavorable"], 9),
]:
    d = h[mask]
    fig.add_scatter(x=d["h1"], y=d["h2"], mode="markers", name=name,
                    marker=dict(symbol="square", size=size, color=color),
                    customdata=np.stack([d["n1"], d["n2"]], axis=-1),
                    hovertemplate="2017 %{x:.0%} (n=%{customdata[0]})<br>"
                                  "2018 %{y:.0%} (n=%{customdata[1]})<extra></extra>")
lo = max(0, min(h["h1"].min(), h["h2"].min()) - 0.02)
fig.add_shape(type="line", x0=lo, y0=lo, x1=1, y1=1, line=dict(color=C["baseline"], width=1, dash="dot"))
fig.add_annotation(x=lo + 0.04, y=lo + 0.04, text="SAME RATE BOTH YEARS", showarrow=False, xanchor="left",
                   yanchor="top", yshift=-6, font=dict(family=MONO, size=10, color=C["axis"]))
fig.update_xaxes(title="Dispatch on time, 2017", tickformat=".0%", range=[lo, 1.01])
fig.update_yaxes(title="Dispatch on time, 2018", tickformat=".0%", range=[lo, 1.01], zeroline=False)
title = (f"Late dispatch is structural: {stay_rate:.0%} of 2017 laggards were still laggards in 2018"
         if stay_rate >= 0.40 else
         f"Late dispatch is mostly circumstantial: only {stay_rate:.0%} of 2017 laggards stayed behind")
finish(fig, title, f"Seller dispatch on-time rate, 2017 vs 2018 · {len(h):,} sellers with 15+ orders in each year · Spearman {rho:.2f}")
fig.show()
save(fig, "04_dispatch_persistence")

WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/04_dispatch_persistence.png')

## 3 · Who causes the delay the customer feels? (Q3)

Every single-seller order with both outcomes known falls into one of four cells. The question for procurement: if suppliers were measured only on the customer delivery date, how many seller failures would go unseen?

In [6]:
d = q(f"""
    SELECT dispatch_on_time, customer_on_time, COUNT(*) AS n
    {BASE} AND dispatch_on_time IS NOT NULL AND customer_on_time IS NOT NULL
    GROUP BY 1, 2""")
n = {(int(r.dispatch_on_time), int(r.customer_on_time)): int(r.n) for r in d.itertuples()}
seller_late = n.get((0, 0), 0) + n.get((0, 1), 0)
seller_ok = n.get((1, 0), 0) + n.get((1, 1), 0)
cust_late = n.get((0, 0), 0) + n.get((1, 0), 0)

p_late_if_seller_late = n.get((0, 0), 0) / seller_late
p_late_if_seller_ok = n.get((1, 0), 0) / seller_ok
masked = n.get((0, 1), 0) / seller_late
seller_share_of_late = n.get((0, 0), 0) / cust_late

matrix = pd.DataFrame(
    [[n.get((1, 1), 0), n.get((1, 0), 0)], [n.get((0, 1), 0), n.get((0, 0), 0)]],
    index=["Seller dispatched on time", "Seller dispatched late"],
    columns=["Customer on time", "Customer late"])
print(matrix.to_string(), "\n")
print(f"Customer late when seller late:     {p_late_if_seller_late:.1%}")
print(f"Customer late when seller on time:  {p_late_if_seller_ok:.1%}  (risk ratio {p_late_if_seller_late / p_late_if_seller_ok:.1f}x)")
print(f"Seller-late orders still on time:   {masked:.1%}  <- invisible if suppliers are measured at delivery")
print(f"Late-to-customer orders where the seller was also late: {seller_share_of_late:.1%}")

                           Customer on time  Customer late
Seller dispatched on time             81692           4709
Seller dispatched late                 6721           1809 

Customer late when seller late:     21.2%
Customer late when seller on time:  5.5%  (risk ratio 3.9x)
Seller-late orders still on time:   78.8%  <- invisible if suppliers are measured at delivery
Late-to-customer orders where the seller was also late: 27.8%


In [7]:
rows = ["Seller on time", "Seller late"]
on_time = [1 - p_late_if_seller_ok, 1 - p_late_if_seller_late]
late = [p_late_if_seller_ok, p_late_if_seller_late]
fig = go.Figure()
fig.add_bar(y=rows, x=on_time, orientation="h", name="Customer received on time",
            marker_color=C["forest_700"], text=[f"{v:.0%}" for v in on_time],
            textposition="inside", insidetextanchor="start", textfont=dict(color=C["page"]))
fig.add_bar(y=rows, x=late, orientation="h", name="Customer received late",
            marker_color=C["unfavorable"], text=[f"{v:.0%}" for v in late],
            textposition="outside", textfont=dict(color=C["unfavorable"]))
fig.update_layout(barmode="stack", bargap=0.45, margin=dict(l=130),
                  legend=dict(traceorder="normal"))
fig.update_xaxes(tickformat=".0%", range=[0, 1.08], title="Share of orders")
fig.update_yaxes(showgrid=False, zeroline=False, autorange="reversed")
finish(fig, f"{masked:.0%} of late seller dispatches still reached the customer on time",
       f"Customer outcome by seller dispatch outcome · single-seller orders · "
       f"customer-late risk {p_late_if_seller_late / p_late_if_seller_ok:.1f}x higher when the seller is late")
fig.show()
save(fig, "05_delay_decomposition", height=520)

WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/05_delay_decomposition.png')

The masking above can only happen if delivery estimates carry slack. The distribution of days between actual delivery and the estimate shows how much.

In [8]:
slack = q("""
    SELECT DATE_DIFF('day', CAST(delivered_customer_at AS DATE), CAST(estimated_delivery_at AS DATE)) AS slack_days
    FROM (SELECT DISTINCT order_id, delivered_customer_at, estimated_delivery_at
          FROM fct_order_items
          WHERE in_window AND is_single_seller AND delivered_customer_at IS NOT NULL
            AND NOT dq_customer_before_carrier)""")["slack_days"]
med_slack = slack.median()
pct_week_early = (slack >= 7).mean()
print(f"Median days delivered before the estimate: {med_slack:.0f}")
print(f"Delivered 7+ days before the estimate:     {pct_week_early:.1%}")
print(f"Delivered after the estimate:              {(slack < 0).mean():.1%}")

clip = slack.clip(-20, 45)
fig = go.Figure()
fig.add_histogram(x=clip[clip < 0], xbins=dict(start=-20.5, end=45.5, size=1), name="After the estimate",
                  marker_color=C["unfavorable"])
fig.add_histogram(x=clip[clip >= 0], xbins=dict(start=-20.5, end=45.5, size=1), name="On or before the estimate",
                  marker_color=C["forest_700"])
fig.add_vline(x=med_slack, line=dict(color=C["ink"], width=1, dash="dash"))
fig.add_annotation(x=med_slack, y=1, yref="paper", text=f"MEDIAN · {med_slack:.0f} DAYS EARLY", showarrow=False,
                   xanchor="left", xshift=6, yanchor="top", font=dict(family=MONO, size=10, color=C["ink"]))
fig.update_layout(barmode="overlay", bargap=0.05)
fig.update_xaxes(title="Days delivered before the estimated date (negative = late; clipped at -20 and 45)")
fig.update_yaxes(title="Orders", tickformat=",.0f")
finish(fig, f"Half of all orders arrive {med_slack:.0f}+ days before the promised date",
       "Distribution of delivery slack vs the estimate shown to the customer · single-seller delivered orders")
fig.show()
save(fig, "06_delivery_slack")

Median days delivered before the estimate: 12
Delivered 7+ days before the estimate:     78.7%
Delivered after the estimate:              6.9%


WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/06_delivery_slack.png')

## 4 · Delay, review and distance (Q4)

In [9]:
r = q("""
    WITH o AS (
        SELECT DISTINCT order_id, review_score,
               DATE_DIFF('day', CAST(estimated_delivery_at AS DATE), CAST(delivered_customer_at AS DATE)) AS days_late
        FROM fct_order_items
        WHERE in_window AND is_single_seller AND delivered_customer_at IS NOT NULL
          AND NOT dq_customer_before_carrier AND review_score IS NOT NULL)
    SELECT CASE WHEN days_late <= 0 THEN '1 · On time'
                WHEN days_late <= 3 THEN '2 · 1-3 days late'
                WHEN days_late <= 7 THEN '3 · 4-7 days late'
                ELSE '4 · 8+ days late' END AS bucket,
           COUNT(*) AS orders,
           AVG(review_score) AS avg_review,
           AVG(CASE WHEN review_score <= 2 THEN 1.0 ELSE 0 END) AS pct_negative
    FROM o GROUP BY 1 ORDER BY 1""")
r["label"] = r["bucket"].str[4:]
print(r[["label", "orders", "avg_review", "pct_negative"]].to_string(index=False))

colors = [C["forest_700"]] + [C["unfavorable"]] * (len(r) - 1)
fig = go.Figure()
fig.add_bar(x=r["label"], y=r["pct_negative"], marker_color=colors,
            text=[f"{v:.0%}<br><span style='font-size:10px'>n={o:,}</span>" for v, o in zip(r["pct_negative"], r["orders"])],
            textposition="outside", textfont=dict(color=C["ink"]), showlegend=False)
fig.update_yaxes(title="Share of reviews with 1 or 2 stars", tickformat=".0%",
                 range=[0, r["pct_negative"].max() * 1.25])
fig.update_xaxes(title="Delivery vs estimated date")
ratio = r["pct_negative"].iloc[-1] / r["pct_negative"].iloc[0]
finish(fig, f"Orders 8+ days late get {ratio:.1f}x the share of negative reviews",
       "Share of 1- and 2-star reviews by delivery delay · single-seller orders with review")
fig.show()
save(fig, "07_reviews_by_delay")

        label  orders  avg_review  pct_negative
      On time   87936       4.311         0.087
1-3 days late    1845       3.295         0.320
4-7 days late    1742       2.104         0.676
 8+ days late    2780       1.697         0.792


WindowsPath('C:/dev/procurement-analytics-olist/reports/figures/07_reviews_by_delay.png')

In [10]:
dist = q("""
    SELECT CASE WHEN distance_km < 100 THEN '1 · under 100 km'
                WHEN distance_km < 500 THEN '2 · 100-499 km'
                WHEN distance_km < 1000 THEN '3 · 500-999 km'
                ELSE '4 · 1,000+ km' END AS band,
           COUNT(*) AS orders,
           MEDIAN(freight_ratio) AS median_freight_ratio,
           1 - AVG(dispatch_on_time) AS seller_late_rate,
           1 - AVG(customer_on_time) AS customer_late_rate
    FROM fct_seller_orders s
    JOIN (SELECT order_id, seller_id, CAST(SUM(freight_value) AS DOUBLE) / NULLIF(CAST(SUM(price) AS DOUBLE), 0) AS freight_ratio
          FROM fct_order_items GROUP BY 1, 2) f USING (order_id, seller_id)
    WHERE s.in_window AND s.is_single_seller AND s.is_valid_status AND s.distance_km IS NOT NULL
    GROUP BY 1 ORDER BY 1""")
dist["band"] = dist["band"].str[4:]
dist.style.format({"orders": "{:,}", "median_freight_ratio": "{:.1%}",
                   "seller_late_rate": "{:.1%}", "customer_late_rate": "{:.1%}"}).hide(axis="index")

band,orders,median_freight_ratio,seller_late_rate,customer_late_rate
under 100 km,"17,937",17.2%,9.6%,4.5%
100-499 km,"36,971",21.3%,8.6%,6.2%
500-999 km,"25,755",24.6%,8.9%,7.2%
"1,000+ km","15,486",29.9%,9.4%,10.5%


Reading the table: the seller's dispatch does not depend on where the customer is, so **seller late rate should be flat across distance bands** while **customer late rate rises with distance**. If both rise, distant sellers are also slower to dispatch, which is a seller problem; if only the customer rate rises, it is a logistics problem.

## 5 · Findings for `docs/findings.md`
Generated from the numbers above. Implications are written by hand after reading them.

In [11]:
near, far = dist.iloc[0], dist.iloc[-1]
seller_flat = abs(far["seller_late_rate"] - near["seller_late_rate"]) < 0.03
customer_rises = far["customer_late_rate"] - near["customer_late_rate"] >= 0.03
if seller_flat and customer_rises:
    dist_head = "**Distance is a logistics problem, not a seller problem.** "
elif seller_flat:
    dist_head = "**Distance barely changes delivery performance.** "
else:
    dist_head = "**Distant sellers are also slower to dispatch.** "
findings = [
    (f"**Seller SLA: {base_dispatch:.1%} of orders dispatched on time in the median month; "
     f"{bf['dispatch_on_time']:.1%} in Nov 2017 (Black Friday, {bf['orders'] / others['orders'].median():.1f}x median volume).**"),
    (f"**Late dispatch is {'structural' if stay_rate >= 0.40 else 'mostly circumstantial'}.** "
     f"{stay_rate:.0%} of sellers in the 2017 bottom quartile were still there in 2018 (25% expected by chance); "
     f"Spearman {rho:.2f} across {len(h):,} sellers."),
    (f"**Measuring suppliers at delivery hides their failures.** {masked:.0%} of orders the seller dispatched late "
     f"still reached the customer on time, because the median order arrives {med_slack:.0f} days before the estimate. "
     f"When the seller is late, the customer is late {p_late_if_seller_late / p_late_if_seller_ok:.1f}x as often."),
    (f"**Lateness drives reviews.** Orders 8+ days late get {ratio:.1f}x the share of 1-2 star reviews of on-time orders "
     f"({r['pct_negative'].iloc[-1]:.0%} vs {r['pct_negative'].iloc[0]:.0%})."),
    dist_head +
    (f"Seller late rate is {near['seller_late_rate']:.1%} under 100 km and {far['seller_late_rate']:.1%} above 1,000 km; "
     f"customer late rate goes from {near['customer_late_rate']:.1%} to {far['customer_late_rate']:.1%}."),
]
for f in findings:
    print("- " + f + "\n")

- **Seller SLA: 92.2% of orders dispatched on time in the median month; 87.0% in Nov 2017 (Black Friday, 1.6x median volume).**

- **Late dispatch is structural.** 62% of sellers in the 2017 bottom quartile were still there in 2018 (25% expected by chance); Spearman 0.59 across 338 sellers.

- **Measuring suppliers at delivery hides their failures.** 79% of orders the seller dispatched late still reached the customer on time, because the median order arrives 12 days before the estimate. When the seller is late, the customer is late 3.9x as often.

- **Lateness drives reviews.** Orders 8+ days late get 9.1x the share of 1-2 star reviews of on-time orders (79% vs 9%).

- **Distance is a logistics problem, not a seller problem.** Seller late rate is 9.6% under 100 km and 9.4% above 1,000 km; customer late rate goes from 4.5% to 10.5%.

